In [0]:
# Databricks notebook source
# Gold Layer - dim_date
# Static calendar dimension. Full overwrite each run is safe and standard practice
# for date dimensions (cheap to regenerate, no upsert logic needed).

from pyspark.sql import functions as F

# Bound the calendar to the actual data range (with a small buffer either side)
date_bounds = (
    spark.table("retail_sales_dev.silver.sales_cleaned")
    .agg(F.min("InvoiceDate").alias("min_date"), F.max("InvoiceDate").alias("max_date"))
    .collect()[0]
)
start_date = date_bounds["min_date"].date()
end_date = date_bounds["max_date"].date()

display(date_bounds)

In [0]:
print(start_date)
print(end_date)

In [0]:
dim_date_df = (
    spark.sql(
        f"""
        SELECT explode(sequence(
            to_date('{start_date}'),
            to_date('{end_date}'),
            interval 1 day
        )) AS CalendarDate
        """
    )
    .withColumn("DateSK", F.date_format("CalendarDate", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("CalendarDate"))
    .withColumn("Quarter", F.quarter("CalendarDate"))
    .withColumn("Month", F.month("CalendarDate"))
    .withColumn("MonthName", F.date_format("CalendarDate", "MMMM"))
    .withColumn("Day", F.dayofmonth("CalendarDate"))
    .withColumn("DayOfWeek", F.dayofweek("CalendarDate"))  # 1 = Sunday ... 7 = Saturday
    .withColumn("DayName", F.date_format("CalendarDate", "EEEE"))
    .withColumn("IsWeekend", F.dayofweek("CalendarDate").isin([1, 7]))
)



In [0]:
display(dim_date_df)

In [0]:

(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dev.gold.dim_date")
)

print(f"dim_date rebuilt: {start_date} -> {end_date} ({dim_date_df.count()} rows)")